In this code, we develop a lang chain based agentic application. Agent get involve with three tools (Google Search, Temperature and a cusotm tool to get the word count)

In [1]:
#install the necessary packages
!pip install langchain -qU
!pip install langchain-openai -qU
!pip install langchain-community -qU

#to do the google search we need to install package duckduckgo-search
!pip install duckduckgo-search -qU
#to get the temperature from openweather.com we need to install package python open weather map
!pip install pyowm -qU

#to take a default prompt, we install langchainhub package
!pip install -U langchain langchainhub


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Initialize the OPENAI LLM

In [2]:
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

import os

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() /'.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)
weather_key=os.getenv("OPEN_WEATHER_API_KEY")
print("Loaded:", weather_key is not None)

Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\.env
File exists: True
Loaded: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#initialize the chatOPENAI model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=api_key)

c:\Users\shara\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialize DuckDuckGo Search Tool.

In Langchain ther have alrdy several builtin tools for different purposes. Duckduck tool is a search tool. Open weathermap tool is for finding temperature. Accordingly we can ue these default tools from the store. 

In [4]:
!pip install -U ddgs
from langchain_community.tools import DuckDuckGoSearchRun
#intiailize search tool
search_tool=DuckDuckGoSearchRun()

search_tool


  Attempting uninstall: ddgs
    Found existing installation: ddgs 9.14.1
    Uninstalling ddgs-9.14.1:
      Successfully uninstalled ddgs-9.14.1



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'))

Initialize the OpenWeatherMap Tool

In [5]:
from langchain_community.utilities import OpenWeatherMapAPIWrapper
from langchain_community.tools.openweathermap import OpenWeatherMapQueryRun

weather = OpenWeatherMapAPIWrapper(openweathermap_api_key=weather_key)
weather_tool = OpenWeatherMapQueryRun(api_wrapper=weather)

weather_tool.run("Melbourne")

'In Melbourne, the current weather is as follows:\nDetailed status: overcast clouds\nWind speed: 1.76 m/s, direction: 180°\nHumidity: 87%\nTemperature: \n  - Current: 23.94°C\n  - High: 23.94°C\n  - Low: 23.94°C\n  - Feels like: 24.66°C\nRain: {}\nHeat index: None\nCloud cover: 100%'

Load a default prompt template from LangChainHub. (We do not bulit prompt of our own, instead we use a defautl prompt form langchian hub)

In [6]:
from langchainhub import client

#get a prompt tempalte from LangChainHub
client = Client()
prompt = client.pull("hwchase17/react")
print(prompt)

NameError: name 'Client' is not defined

Create the bulit-in Agent

In [ ]:
#here we create a ommon agent, This agent can work approprately withotu our involement
from langchain.agents import create_agent

#list the tools under this agent
tools=[search_tool,weather_tool]

#create the agent using the LLM and the prompt template
agent=create_agent(tools=tools,model=llm, system_prompt=prompt)

In [ ]:
#now chck how agent manage with user requests. 
#when use ask for a temperature in city, it shold manage to use the temperaure tool and tell us about the weather
response = agent.invoke({"messages": [{"role": "user", "content": "Search the weather in Galle"}]})


#import pretty print to print the response line by line
from pprint import pprint
pprint(response)

print(response["messages"][-1].content)

In [ ]:
# Now We ask about the ML. now agent tool should se the search tool
response2= agent.invoke({"messages": [{"role": "user", "content": "What is machine learning"}]})
print(response2["messages"][-1].content)

In [ ]:
#now let we ask it to do a work which not possible with currently avaiable tools.in that case it connot give  correct answer, it just use avable tools. 
response3=agent.invoke({"messages": [{"role": "user", "content": "How many words count within intorduction to machine learning sentence"}]})
print(response3["messages"][-1].content)

Defining a simple custom toolf ro word couting using langchain

In [ ]:
from langchain_core.tools import tool

@tool #a decorator
def word_count(text: str) -> int:
    """Return the number of words in the given text."""
    return len(text.split())

In [ ]:
#add the new custom tool to exisitng tool list
tools.append(word_count)
print(tools)

In [ ]:
#crerate the new agent with new tool list
agent_new=create_agent(tools=tools,model=llm, system_prompt=prompt)

In [ ]:
# Now We ask about the ML. now agent tool should se the search tool
response2_new= agent_new.invoke({"messages": [{"role": "user", "content": "What is machine learning"}]})
print(response2_new["messages"][-1].content)

In [ ]:
#now let we ask it to do a work which not possible with currently avaiable tools.in that case it connot give  correct answer, it just use avable tools. 
response3_new=agent_new.invoke({"messages": [{"role": "user", "content": "How many words count wihtin intorduction to machine learning"}]})
print(response3_new["messages"][-1].content)

In [ ]:
#now let we ask it to do work with two tools (search tool and temperature tool)
response4_new=agent_new.invoke({"messages": [{"role": "user", "content": "What is the current temperature of city which is Sigirya located"}]})
print(response4_new["messages"][-1].content)